In [4]:
import os
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm


from typing import List, Dict  

def parse_single_annotation(xml_path: str, images_dir: str) -> List[Dict]:
    """Парсит один XML-файл аннотации"""
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        filename = root.find('filename').text
        image_id = os.path.splitext(filename)[0]
        image_path = os.path.join(images_dir, filename)
        
        size = root.find('size')
        width = int(size.find('width').text)
        height = int(size.find('height').text)
        
        annotations = []
        
        for obj in root.findall('object'):
            label = obj.find('name').text
            bndbox = obj.find('bndbox')
            
            annotations.append({
                'img_id': image_id,
                'label': label,
                'bbox': [
                    [int(bndbox.find('xmin').text), int(bndbox.find('ymin').text)],
                    [int(bndbox.find('xmax').text), int(bndbox.find('ymax').text)]
                ],
                'Image_path': image_path
            })
            
        return annotations
        
    except Exception as e:
        print(f"Error parsing {xml_path}: {str(e)}")
        return []

def parse_all_annotations(annotations_dir: str, images_dir: str) -> pd.DataFrame:
    """Парсит все XML-файлы в директории"""
    all_data = []
    xml_files = [f for f in os.listdir(annotations_dir) if f.endswith('.xml')]
    
    for xml_file in tqdm(xml_files, desc="Processing XML files"):
        xml_path = os.path.join(annotations_dir, xml_file)
        annotations = parse_single_annotation(xml_path, images_dir)
        all_data.extend(annotations)
    
    return pd.DataFrame(all_data, columns=['img_id', 'label', 'bbox', 'Image_path'])

# Пути к данным
annotations_dir = 'Dataset5/archive/LP-characters/annotations'
images_dir = 'Dataset5/archive/LP-characters/images'

# Парсим все аннотации
df = parse_all_annotations(annotations_dir, images_dir)

# Проверяем результат
print(f"Total annotations: {len(df)}")
print(df.head())

# Сохраняем в CSV
df.to_csv('d5.csv', index=False)


# нужно разделить этот df на train и test

Processing XML files: 100%|██████████| 209/209 [00:00<00:00, 5240.59it/s]

Total annotations: 2026
  img_id label                  bbox  \
0   0059     H   [[7, 33], [20, 53]]   
1   0059     R  [[20, 33], [32, 53]]   
2   0059     9  [[32, 33], [42, 53]]   
3   0059     9  [[42, 33], [53, 53]]   
4   0059     E  [[58, 33], [69, 53]]   

                                       Image_path  
0  Dataset5/archive/LP-characters/images/0059.png  
1  Dataset5/archive/LP-characters/images/0059.png  
2  Dataset5/archive/LP-characters/images/0059.png  
3  Dataset5/archive/LP-characters/images/0059.png  
4  Dataset5/archive/LP-characters/images/0059.png  


In [6]:
import os
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split


from typing import List, Dict  

def parse_single_annotation(xml_path: str, images_dir: str) -> List[Dict]:
    """Парсит один XML-файл аннотации"""
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        filename = root.find('filename').text
        image_id = os.path.splitext(filename)[0]
        image_path = os.path.join(images_dir, filename)
        
        size = root.find('size')
        width = int(size.find('width').text)
        height = int(size.find('height').text)
        
        annotations = []
        
        for obj in root.findall('object'):
            label = obj.find('name').text
            bndbox = obj.find('bndbox')
            
            annotations.append({
                'img_id': image_id,
                'label': label,
                'bbox': [
                    [int(bndbox.find('xmin').text), int(bndbox.find('ymin').text)],
                    [int(bndbox.find('xmax').text), int(bndbox.find('ymax').text)]
                ],
                'Image_path': image_path
            })
            
        return annotations
        
    except Exception as e:
        print(f"Error parsing {xml_path}: {str(e)}")
        return []

def parse_all_annotations(annotations_dir: str, images_dir: str) -> pd.DataFrame:
    """Парсит все XML-файлы в директории"""
    all_data = []
    xml_files = [f for f in os.listdir(annotations_dir) if f.endswith('.xml')]
    
    for xml_file in tqdm(xml_files, desc="Processing XML files"):
        xml_path = os.path.join(annotations_dir, xml_file)
        annotations = parse_single_annotation(xml_path, images_dir)
        all_data.extend(annotations)
    
    return pd.DataFrame(all_data, columns=['img_id', 'label', 'bbox', 'Image_path'])

# Пути к данным
annotations_dir = 'Dataset5/archive/LP-characters/annotations'
images_dir = 'Dataset5/archive/LP-characters/images'

# Парсим все аннотации
df = parse_all_annotations(annotations_dir, images_dir)

# Проверяем результат
print(f"Total annotations: {len(df)}")
print(df.head())

# Разделяем на train и test в соотношении 5:1
train_df, test_df = train_test_split(df, test_size=0.1667, random_state=42)

# Проверяем размеры
print(f"Train annotations: {len(train_df)}")
print(f"Test annotations: {len(test_df)}")

# Сохраняем в CSV
train_df.to_csv('d5_train.csv', index=False)
test_df.to_csv('d5_test.csv', index=False)

print("Train dataset saved to 'd5_train.csv'")
print("Test dataset saved to 'd5_test.csv'")


df.head(25) 

Processing XML files: 100%|██████████| 209/209 [00:00<00:00, 6299.75it/s]

Total annotations: 2026
  img_id label                  bbox  \
0   0059     H   [[7, 33], [20, 53]]   
1   0059     R  [[20, 33], [32, 53]]   
2   0059     9  [[32, 33], [42, 53]]   
3   0059     9  [[42, 33], [53, 53]]   
4   0059     E  [[58, 33], [69, 53]]   

                                       Image_path  
0  Dataset5/archive/LP-characters/images/0059.png  
1  Dataset5/archive/LP-characters/images/0059.png  
2  Dataset5/archive/LP-characters/images/0059.png  
3  Dataset5/archive/LP-characters/images/0059.png  
4  Dataset5/archive/LP-characters/images/0059.png  
Train annotations: 1688
Test annotations: 338
Train dataset saved to 'd5_train.csv'
Test dataset saved to 'd5_test.csv'


,img_id,label,bbox,Image_path
0,0059,H,"[[7, 33], [20, 53]]",Dataset5/archive/LP-characters/images/0059.png
1,0059,R,"[[20, 33], [32, 53]]",Dataset5/archive/LP-characters/images/0059.png
2,0059,9,"[[32, 33], [42, 53]]",Dataset5/archive/LP-characters/images/0059.png
3,0059,9,"[[42, 33], [53, 53]]",Dataset5/archive/LP-characters/images/0059.png
4,0059,E,"[[58, 33], [69, 53]]",Dataset5/archive/LP-characters/images/0059.png
5,0059,X,"[[69, 33], [82, 54]]",Dataset5/archive/LP-characters/images/0059.png
6,0059,T,"[[92, 33], [103, 54]]",Dataset5/archive/LP-characters/images/0059.png
7,0059,E,"[[103, 33], [115, 53]]",Dataset5/archive/LP-characters/images/0059.png
8,0059,M,"[[115, 34], [129, 54]]",Dataset5/archive/LP-characters/images/0059.png
9,0059,P,"[[129, 34], [140, 54]]",Dataset5/archive/LP-characters/images/0059.png
